# Notebook 03 v2 — NSL-KDD calibration (hybrid Platt/isotonic)

**What this notebook does:**

Fits per-class calibrators on the calibration set carved out in Notebook 01 v2 (M6 fix), applies them to the test set, and reports per-class calibration quality.

**Calibration strategy (decided in response to TA's rare-class caveat comment):**

- **Isotonic regression** for classes with n_calib ≥ 30 (parameter-rich, flexible monotonic fit)
- **Platt scaling** (sigmoid, 2 parameters) for classes with n_calib < 30 (parameter-efficient)

NSL calibration set sizes:
- Normal: 13,469 → isotonic
- DoS: 9,186 → isotonic
- Probe: 2,331 → isotonic
- R2L: 199 → isotonic
- **U2R: 10 → Platt** (the only rare class needing Platt)

**Per-class metrics reported:**

- **Brier score**: lower is better, measures probabilistic accuracy
- **ECE (Expected Calibration Error)**: lower is better, measures how well predicted probabilities match observed frequencies
- **Strategy used**: explicit per-class recording of isotonic vs Platt

**M6 fix:** Calibrators fit on `calib_proba.npy` (from training-data-derived calibration set), applied to `test_proba.npy` (official untouched test set).

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

!git pull
print(f'\n✓ Ready in: {os.getcwd()}')

In [ ]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
from collections import Counter

from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

SEED = 42
np.random.seed(SEED)

DATASET = 'nsl_kdd_v2'
PROCESSED = Path(REPO) / 'data' / 'processed' / DATASET
MODELS_DIR = Path(REPO) / 'models' / DATASET
PREDS_DIR = MODELS_DIR / 'predictions'
CALIB_OUT_DIR = Path(REPO) / 'calibrators' / DATASET
CALIB_OUT_DIR.mkdir(parents=True, exist_ok=True)

# Decision threshold for Platt vs isotonic
PLATT_THRESHOLD = 30

# Number of bins for ECE computation
ECE_N_BINS = 10

print(f'Dataset: {DATASET}')
print(f'Platt threshold: n_calib < {PLATT_THRESHOLD} → Platt; n_calib ≥ {PLATT_THRESHOLD} → isotonic')
print(f'Output dir: {CALIB_OUT_DIR}')

## 2. Load calibration set labels and class mappings

In [ ]:
# Labels for the calibration set (used to fit calibrators)
y_calib_b = np.load(PROCESSED / 'y_calib_binary.npy')
y_calib_5 = np.load(PROCESSED / 'y_calib_5class.npy')

# Labels for the test set (used to evaluate calibrated probabilities)
y_test_b = np.load(PROCESSED / 'y_test_binary.npy')
y_test_5 = np.load(PROCESSED / 'y_test_5class.npy')

with open(PROCESSED / 'class_mappings.json') as f:
    class_info = json.load(f)
INT_TO_CATEGORY = {int(k): v for k, v in class_info['multiclass_5'].items()}
CLASS_NAMES_5 = [INT_TO_CATEGORY[i] for i in range(5)]

print(f'Calibration set: {len(y_calib_b):,} samples')
print(f'Test set: {len(y_test_b):,} samples')
print()

# Per-class calibration sizes
calib_counts_5 = Counter(y_calib_5)
calib_counts_b = Counter(y_calib_b)

print('Per-class calibration set sizes:')
print(f'  Binary:')
for c in [0, 1]:
    n = calib_counts_b[c]
    strat = 'Platt' if n < PLATT_THRESHOLD else 'isotonic'
    print(f'    {"Normal" if c == 0 else "Attack":8s}: n={n:>6,}  → {strat}')

print(f'  5-class:')
for c in range(5):
    n = calib_counts_5[c]
    strat = 'Platt' if n < PLATT_THRESHOLD else 'isotonic'
    print(f'    {CLASS_NAMES_5[c]:8s}: n={n:>6,}  → {strat}')

## 3. Helper functions

In [ ]:
def fit_calibrator(p_calib, y_indicator, n_class):
    """Fit either isotonic or Platt depending on calibration sample size.
    
    Args:
        p_calib: raw probabilities for class c on calibration set, shape (n_calib,)
        y_indicator: 1[y_calib == c] for class c, shape (n_calib,) 
        n_class: number of calibration samples in class c
    
    Returns:
        (calibrator, strategy_name)
    """
    if n_class >= PLATT_THRESHOLD:
        # Isotonic regression: monotonic stepwise fit
        cal = IsotonicRegression(out_of_bounds='clip')
        cal.fit(p_calib, y_indicator)
        return cal, 'isotonic'
    else:
        # Platt scaling: logistic regression on the single feature p_calib
        cal = LogisticRegression(C=1e10, solver='lbfgs')  # ~unregularised sigmoid
        cal.fit(p_calib.reshape(-1, 1), y_indicator)
        return cal, 'platt'

def apply_calibrator(calibrator, strategy, p_test):
    """Apply fitted calibrator to test probabilities."""
    if strategy == 'isotonic':
        return calibrator.predict(p_test)
    else:  # platt
        return calibrator.predict_proba(p_test.reshape(-1, 1))[:, 1]

def expected_calibration_error(probs, labels, n_bins=10):
    """Compute ECE: weighted average gap between accuracy and confidence per bin.
    
    For multiclass, this is the standard formulation using predicted-class confidence.
    For per-class one-vs-rest, probs and labels are binary (P[y=c], 1[y=c]).
    """
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    n = len(probs)
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        if i == n_bins - 1:
            mask = (probs >= lo) & (probs <= hi)
        else:
            mask = (probs >= lo) & (probs < hi)
        if mask.sum() == 0:
            continue
        bin_conf = probs[mask].mean()
        bin_acc = labels[mask].mean()
        weight = mask.sum() / n
        ece += weight * abs(bin_conf - bin_acc)
    return float(ece)

def calibrate_model_per_class(p_calib_2d, y_calib, p_test_2d, y_test, n_classes):
    """Apply per-class calibration with hybrid Platt/isotonic.
    
    Args:
        p_calib_2d: raw probs on calib set, shape (n_calib, n_classes)
        y_calib: integer labels for calib set, shape (n_calib,)
        p_test_2d: raw probs on test set, shape (n_test, n_classes)
        y_test: integer labels for test set, shape (n_test,)
        n_classes: number of classes
    
    Returns:
        dict with calibrated probs, per-class strategies, per-class Brier and ECE
    """
    p_test_cal = np.zeros_like(p_test_2d)
    strategies = {}
    brier_pre = {}
    brier_post = {}
    ece_pre = {}
    ece_post = {}
    
    calib_counts = Counter(y_calib)
    
    for c in range(n_classes):
        # One-vs-rest indicator vectors
        y_calib_c = (y_calib == c).astype(int)
        y_test_c = (y_test == c).astype(int)
        p_calib_c = p_calib_2d[:, c]
        p_test_c = p_test_2d[:, c]
        
        n_c = calib_counts[c]
        cal, strat = fit_calibrator(p_calib_c, y_calib_c, n_c)
        p_test_cal[:, c] = apply_calibrator(cal, strat, p_test_c)
        strategies[c] = strat
        
        # Per-class Brier (pre-cal and post-cal)
        brier_pre[c]  = float(brier_score_loss(y_test_c, p_test_c))
        brier_post[c] = float(brier_score_loss(y_test_c, p_test_cal[:, c]))
        
        # Per-class ECE
        ece_pre[c]  = expected_calibration_error(p_test_c,  y_test_c, ECE_N_BINS)
        ece_post[c] = expected_calibration_error(p_test_cal[:, c], y_test_c, ECE_N_BINS)
    
    # Renormalise so each row sums to 1 (calibrated probs may not sum to 1 naturally)
    row_sums = p_test_cal.sum(axis=1, keepdims=True)
    row_sums = np.where(row_sums == 0, 1, row_sums)  # avoid div by zero
    p_test_cal_renorm = p_test_cal / row_sums
    
    return {
        'p_test_cal_raw': p_test_cal,           # before renormalisation
        'p_test_cal': p_test_cal_renorm,        # after renormalisation
        'strategies': strategies,
        'brier_pre': brier_pre,
        'brier_post': brier_post,
        'ece_pre': ece_pre,
        'ece_post': ece_post,
    }

def calibrate_binary(p_calib_2d, y_calib, p_test_2d, y_test):
    """For binary models: calibrate only P[class=1] (attack probability).
    
    Binary 5-class doesn't apply here; this is for the binary-task models specifically.
    """
    p_calib_attack = p_calib_2d[:, 1]
    p_test_attack = p_test_2d[:, 1]
    y_calib_attack = y_calib  # already 0/1
    y_test_attack = y_test
    
    calib_counts = Counter(y_calib)
    n_attack = calib_counts[1]
    
    cal, strat = fit_calibrator(p_calib_attack, y_calib_attack, n_attack)
    p_test_cal_attack = apply_calibrator(cal, strat, p_test_attack)
    p_test_cal_attack = np.clip(p_test_cal_attack, 0, 1)
    
    # 2-column form for consistency: [P[normal], P[attack]]
    p_test_cal_2d = np.column_stack([1 - p_test_cal_attack, p_test_cal_attack])
    
    return {
        'p_test_cal': p_test_cal_2d,
        'strategies': {1: strat},  # only the attack class was calibrated
        'brier_pre':  {1: float(brier_score_loss(y_test_attack, p_test_attack))},
        'brier_post': {1: float(brier_score_loss(y_test_attack, p_test_cal_attack))},
        'ece_pre':  {1: expected_calibration_error(p_test_attack, y_test_attack, ECE_N_BINS)},
        'ece_post': {1: expected_calibration_error(p_test_cal_attack, y_test_attack, ECE_N_BINS)},
    }

print('✓ Helper functions loaded')

## 4. Calibrate all 9 models

In [ ]:
ALL_MODELS = [
    ('rf_binary_cw',     'binary'),
    ('xgb_binary_cw',    'binary'),
    ('dnn_binary_cw',    'binary'),
    ('rf_5class_smote',  '5class'),
    ('xgb_5class_smote', '5class'),
    ('dnn_5class_smote', '5class'),
    ('rf_5class_cw',     '5class'),
    ('xgb_5class_cw',    '5class'),
    ('dnn_5class_cw',    '5class'),
]

calibration_summary = {}

for name, task in ALL_MODELS:
    print(f'\nCalibrating {name} ({task})...')
    
    p_calib = np.load(PREDS_DIR / f'{name}_calib_proba.npy')
    p_test = np.load(PREDS_DIR / f'{name}_test_proba.npy')
    
    # Ensure 2D shape
    if p_calib.ndim == 1:
        p_calib = np.column_stack([1 - p_calib, p_calib])
    if p_test.ndim == 1:
        p_test = np.column_stack([1 - p_test, p_test])
    
    if task == 'binary':
        result = calibrate_binary(p_calib, y_calib_b, p_test, y_test_b)
    else:  # 5class
        result = calibrate_model_per_class(p_calib, y_calib_5, p_test, y_test_5, n_classes=5)
    
    # Save calibrated probabilities
    np.save(CALIB_OUT_DIR / f'{name}_test_proba_calibrated.npy', result['p_test_cal'])
    
    # Report per-class summary
    print(f'  Strategies: {result["strategies"]}')
    if task == '5class':
        for c in range(5):
            cname = CLASS_NAMES_5[c]
            print(f'  {cname:8s}: Brier {result["brier_pre"][c]:.4f} → {result["brier_post"][c]:.4f}'
                  f'   ECE {result["ece_pre"][c]:.4f} → {result["ece_post"][c]:.4f}'
                  f'   [{result["strategies"][c]}]')
    else:
        print(f'  Attack  : Brier {result["brier_pre"][1]:.4f} → {result["brier_post"][1]:.4f}'
              f'   ECE {result["ece_pre"][1]:.4f} → {result["ece_post"][1]:.4f}'
              f'   [{result["strategies"][1]}]')
    
    calibration_summary[name] = {
        'task': task,
        'strategies': {int(k): v for k, v in result['strategies'].items()},
        'brier_pre':  {int(k): v for k, v in result['brier_pre'].items()},
        'brier_post': {int(k): v for k, v in result['brier_post'].items()},
        'ece_pre':  {int(k): v for k, v in result['ece_pre'].items()},
        'ece_post': {int(k): v for k, v in result['ece_post'].items()},
    }

# Save summary
with open(CALIB_OUT_DIR / 'calibration_summary.json', 'w') as f:
    json.dump(calibration_summary, f, indent=2)

print(f'\n✓ All 9 models calibrated. Summary saved to {CALIB_OUT_DIR}/calibration_summary.json')

## 5. Summary table

In [ ]:
# Build a paper-ready summary table
rows = []
for name, task in ALL_MODELS:
    s = calibration_summary[name]
    if task == '5class':
        # Macro-average Brier and ECE across the 5 classes
        brier_pre_macro = np.mean(list(s['brier_pre'].values()))
        brier_post_macro = np.mean(list(s['brier_post'].values()))
        ece_pre_macro = np.mean(list(s['ece_pre'].values()))
        ece_post_macro = np.mean(list(s['ece_post'].values()))
        platt_classes = [CLASS_NAMES_5[c] for c, st in s['strategies'].items() if st == 'platt']
        rows.append({
            'Model': name,
            'Task': '5-class',
            'Brier pre (macro)':  round(brier_pre_macro,  5),
            'Brier post (macro)': round(brier_post_macro, 5),
            'ECE pre (macro)':    round(ece_pre_macro,    5),
            'ECE post (macro)':   round(ece_post_macro,   5),
            'Platt for': ','.join(platt_classes) if platt_classes else '—',
        })
    else:
        rows.append({
            'Model': name,
            'Task': 'binary',
            'Brier pre (macro)':  round(s['brier_pre'][1],  5),
            'Brier post (macro)': round(s['brier_post'][1], 5),
            'ECE pre (macro)':    round(s['ece_pre'][1],    5),
            'ECE post (macro)':   round(s['ece_post'][1],   5),
            'Platt for': '—' if s['strategies'][1] == 'isotonic' else 'attack class',
        })

df = pd.DataFrame(rows)
print('\nNSL v2 — Hybrid Platt/Isotonic Calibration Results')
print('=' * 110)
print(df.to_string(index=False))
print('=' * 110)

# Save as paper-ready CSV
TABLES_DIR = Path(REPO) / 'results' / 'tables'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
csv_path = TABLES_DIR / 'nslkdd_v2_calibration_summary.csv'
df.to_csv(csv_path, index=False)
print(f'\nSaved: {csv_path}')

## 6. Per-class breakdown (5-class models only)

In [ ]:
# Detailed per-class table for the 5-class models
perclass_rows = []
for name, task in ALL_MODELS:
    if task != '5class':
        continue
    s = calibration_summary[name]
    for c in range(5):
        perclass_rows.append({
            'Model': name,
            'Class': CLASS_NAMES_5[c],
            'n_calib': calib_counts_5[c],
            'Strategy': s['strategies'][c],
            'Brier pre':  round(s['brier_pre'][c],  5),
            'Brier post': round(s['brier_post'][c], 5),
            'ECE pre':    round(s['ece_pre'][c],    5),
            'ECE post':   round(s['ece_post'][c],   5),
        })

df_perclass = pd.DataFrame(perclass_rows)
print('\nNSL v2 — Per-class calibration breakdown')
print('=' * 110)
print(df_perclass.to_string(index=False))
print('=' * 110)

csv_path = TABLES_DIR / 'nslkdd_v2_calibration_perclass.csv'
df_perclass.to_csv(csv_path, index=False)
print(f'\nSaved: {csv_path}')

## 7. Sanity checks

In [ ]:
# Verify that calibration didn't break anything
print('Sanity checks on calibrated probabilities:')
print('-' * 70)

for name, task in ALL_MODELS:
    p_cal = np.load(CALIB_OUT_DIR / f'{name}_test_proba_calibrated.npy')
    
    # Check 1: shape
    if task == 'binary':
        expected_cols = 2
    else:
        expected_cols = 5
    shape_ok = p_cal.shape[1] == expected_cols
    
    # Check 2: range [0, 1]
    range_ok = (p_cal >= 0).all() and (p_cal <= 1).all()
    
    # Check 3: rows sum to ~1 (within 0.001)
    row_sums = p_cal.sum(axis=1)
    sum_ok = np.allclose(row_sums, 1, atol=0.001)
    max_dev = float(np.abs(row_sums - 1).max())
    
    # Check 4: no NaN or inf
    finite_ok = np.isfinite(p_cal).all()
    
    all_ok = shape_ok and range_ok and sum_ok and finite_ok
    flag = '✓' if all_ok else '✗'
    print(f'  {flag} {name:<22} shape={p_cal.shape}  range_ok={range_ok}  '
          f'sum_max_dev={max_dev:.5f}  finite={finite_ok}')

print('\nAll calibrated outputs validated.')

## 8. Commit

In [ ]:
os.chdir(REPO)
!git add notebooks/03_nsl_calibration_v2.ipynb
!git add calibrators/nsl_kdd_v2/
!git add results/tables/nslkdd_v2_calibration_summary.csv
!git add results/tables/nslkdd_v2_calibration_perclass.csv
!git status --short
!git commit -m 'Notebook 03-NSL v2: hybrid Platt/isotonic calibration with per-class Brier and ECE'
!git push origin main